**Name:** Thiều Quang  
**ID:** 24127114  
**Thành viên:** Tôn Thất Việt Khoa — 24127190  
**Môn học:** Khai thác dữ liệu và ứng dụng (*Data Mining and Applications*)  
**Đề bài:** [HCMUS — Khai thác tập mục phổ biến](https://hcmus-cs-ta.github.io/lab-book/data-mining/projects/itemset/)

# Đồ án Eclat — Notebook thực nghiệm

**Phiên bản:** LOCAL — VS Code / IDE

Dùng Python kernel trên máy và mã nguồn trong thư mục project. Không tải ZIP, không clone GitHub trong notebook.

**Nội dung:** ECLAT Python/Julia, kiểm thử, benchmark, SPMF, Market Basket, Visual Patterns.

**Yêu cầu môi trường:** Python 3.13, Java 25 trở lên (cho SPMF), Julia (cho Bước 7B), kết nối mạng ở lần chạy đầu để tải dữ liệu. Chạy lần lượt từ trên xuống (*Run All*); nếu một bước lỗi thì dừng và sửa trước khi chạy tiếp.


## Bước 1 — Thiết lập mã nguồn LOCAL

Mở thư mục gốc của repo bằng VS Code, chọn Python kernel đã cài. Cell này tự tìm thư mục dự án, không gọi Google Colab.


In [1]:
from pathlib import Path
import os
import sys

home = Path.home()
candidates = [
    Path.cwd(), *Path.cwd().parents,
    home / 'Downloads' / 'Group_9_Final',
    home / 'Downloads' / 'Group_9-Frequent-Itemset-Mining'
]

def find_project() -> Path:
    for candidate in candidates:
        if ((candidate / 'src/algorithm/eclat.py').is_file()
                and (candidate / 'julia/EclatJulia.jl').is_file()):
            return candidate.resolve()
    raise FileNotFoundError('Không tìm thấy project ECLAT. Hãy mở thư mục gốc của repo bằng VS Code.')

project: Path = find_project()
os.chdir(project)

if str(project) not in sys.path:
    sys.path.insert(0, str(project))

print('LOCAL project:', project)
print('Python kernel:', sys.executable)


LOCAL project: D:\gg\Group_9-Frequent-Itemset-Mining-main
Python kernel: c:\Users\aer1al\AppData\Local\Programs\Python\Python312\python.exe


In [2]:
import shutil
import subprocess

def tool_version(cmd):
    exe = shutil.which(cmd)
    if not exe:
        return None
    r = subprocess.run([exe, '--version' if cmd != 'java' else '-version'],
                       capture_output=True, text=True)
    return (r.stdout + r.stderr).strip().splitlines()[0]

print('Python :', sys.version.split()[0])
for tool in ('java', 'julia'):
    info = tool_version(tool)
    print(f'{tool:7}:', info if info else 'CHƯA CÓ trên PATH')
for rel in ('requirements.txt', 'src/algorithm/eclat.py', 'julia/EclatJulia.jl', 'scripts', 'data'):
    print(('OK   ' if (project / rel).exists() else 'THIẾU'), rel)


Python : 3.12.10
java   : openjdk version "25.0.5.1" 2026-11-17 LTS
julia  : CHƯA CÓ trên PATH
OK    requirements.txt
OK    src/algorithm/eclat.py
OK    julia/EclatJulia.jl
OK    scripts
OK    data


## Bước 2 — Cài thư viện và chạy kiểm thử

`pytest` là kiểm tra nội bộ trên tập toy; **không thay thế** bước so với SPMF. Nếu test lỗi thì dừng để sửa, không sử dụng kết quả benchmark.

In [3]:
import subprocess

def run_module(name, *args):
    command = [sys.executable, '-m', name, *map(str, args)]
    print('>>>', ' '.join(command), flush=True)
    subprocess.run(command, cwd=project, check=True)

subprocess.run([sys.executable,'-m','pip','-q','install','-r','requirements.txt'],check=True)
run_module('pytest','-q')
run_module('src.cli','data/toy/baskets.txt','--minsup','0.5','--backend','bitset','--output','results/toy_result.txt')
print((project/'results/toy_result.txt').read_text(encoding='utf-8'))

>>> c:\Users\aer1al\AppData\Local\Programs\Python\Python312\python.exe -m pytest -q
>>> c:\Users\aer1al\AppData\Local\Programs\Python\Python312\python.exe -m src.cli data/toy/baskets.txt --minsup 0.5 --backend bitset --output results/toy_result.txt
1 #SUP: 4
2 #SUP: 5
3 #SUP: 4
1 2 #SUP: 3
1 3 #SUP: 3
2 3 #SUP: 3



## Bước 3 — Tải dữ liệu benchmark và dữ liệu ứng dụng

Chương trình dùng các URL được ghi trong `DATA_SOURCES.md`, lưu manifest kèm SHA-256. Nếu mạng nguồn lỗi, bạn có thể tải thủ công tại [SPMF Datasets](https://www.philippe-fournier-viger.com/spmf/index.php?link=datasets.php) và [UCI Optdigits](https://archive.ics.uci.edu/dataset/80/optical+recognition+of+handwritten+digits), rồi đặt đúng tên file vào thư mục `data/` (xem tài liệu nguồn). Không được tự thay bằng dữ liệu tạo ngẫu nhiên rồi gọi là benchmark thật.

In [4]:
run_module('scripts.download_data')
run_module('scripts.dataset_statistics')
print((project/'data/download_manifest.json').read_text(encoding='utf-8')[:1600])

>>> c:\Users\aer1al\AppData\Local\Programs\Python\Python312\python.exe -m scripts.download_data
>>> c:\Users\aer1al\AppData\Local\Programs\Python\Python312\python.exe -m scripts.dataset_statistics
{
  "foodmart": {
    "source_url": "https://www.philippe-fournier-viger.com/spmf/publicdatasets/foodmartFIM.txt",
    "acquisition_method": "existing_with_matching_manifest",
    "description": "SPMF public FIM dataset, real retail",
    "local_path": "data/benchmark/foodmart.txt",
    "size_bytes": 82840,
    "sha256": "8762f2000459e94ee166bd813763567b2b60dfb24970e1cffec497b23a694081",
    "recorded_utc": "2026-11-26T14:12:40.886272+00:00"
  },
  "c20d10k": {
    "source_url": "https://www.philippe-fournier-viger.com/spmf/publicdatasets/c20d10k.txt",
    "acquisition_method": "existing_with_matching_manifest",
    "description": "SPMF synthetic FIM dataset",
    "local_path": "data/benchmark/c20d10k.txt",
    "size_bytes": 800000,
    "sha256": "1ec8dca5aedaad57239481b726f67bf11cd8f51532aad

## Bước 4 — Chạy benchmark trên bốn bộ dữ liệu

Chạy Eclat `set` và `bitset`. Output: `results/benchmark.csv` và `results/rss_memory.csv`. `status=invalid` nghĩa là phép đo chưa hoàn tất, không được báo cáo như kết quả hợp lệ. **Nếu hết RAM**, giảm tải minh bạch, điều chỉnh minsup trong script và ghi thay đổi vào báo cáo; không tự ghi số liệu.

In [5]:
run_module('scripts.benchmark')
run_module('scripts.rss_memory')
run_module('scripts.scalability')
run_module('scripts.length_experiment')
print('Đã tạo benchmark.csv, rss_memory.csv, scalability.csv, length_experiment.csv')

>>> c:\Users\aer1al\AppData\Local\Programs\Python\Python312\python.exe -m scripts.benchmark
>>> c:\Users\aer1al\AppData\Local\Programs\Python\Python312\python.exe -m scripts.rss_memory
>>> c:\Users\aer1al\AppData\Local\Programs\Python\Python312\python.exe -m scripts.scalability
>>> c:\Users\aer1al\AppData\Local\Programs\Python\Python312\python.exe -m scripts.length_experiment
Đã tạo benchmark.csv, rss_memory.csv, scalability.csv, length_experiment.csv


In [6]:
import csv

def preview(name, n=5):
    path = project / 'results' / name
    with path.open(encoding='utf-8-sig', newline='') as f:
        rows = list(csv.DictReader(f))
    print(f'\n{name}: {len(rows)} dòng, cột = {list(rows[0].keys()) if rows else []}')
    for r in rows[:n]:
        print('  ', dict(r))

for name in ('benchmark.csv', 'rss_memory.csv', 'scalability.csv', 'length_experiment.csv'):
    preview(name)



benchmark.csv: 48 dòng, cột = ['dataset', 'dataset_sha256', 'transactions', 'unique_items', 'max_singleton_support', 'minsup', 'min_count', 'minsup_percentage', 'backend', 'seconds', 'peak_python_mb', 'itemset_count', 'status', 'error', 'baseline_match', 'python_version', 'platform']
   {'dataset': 'foodmart', 'dataset_sha256': '8762f2000459e94ee166bd813763567b2b60dfb24970e1cffec497b23a694081', 'transactions': '4141', 'unique_items': '1559', 'max_singleton_support': '25', 'minsup': '25', 'min_count': '25', 'minsup_percentage': '0.60371891', 'backend': 'set', 'seconds': '0.015276', 'peak_python_mb': '3.1666', 'itemset_count': '1', 'status': 'ok', 'error': '', 'baseline_match': 'baseline', 'python_version': '3.12.10', 'platform': 'Windows-11-10.0.26200-SP0'}
   {'dataset': 'foodmart', 'dataset_sha256': '8762f2000459e94ee166bd813763567b2b60dfb24970e1cffec497b23a694081', 'transactions': '4141', 'unique_items': '1559', 'max_singleton_support': '25', 'minsup': '25', 'min_count': '25', 'mins

## Bước 5 — Đối chiếu từng tập phổ biến và support với SPMF

SPMF chỉ chạy như **phần mềm tham chiếu Java**, không dùng thư viện hoặc mã nguồn của nó để cài Eclat. Chương trình thử tải release `.jar` từ trang chính thức. Nếu bị lỗi mạng, tải file `spmf.jar` từ [SPMF Download](https://www.philippe-fournier-viger.com/spmf/index.php?link=download.php) rồi đặt vào `tools/spmf.jar`.

In [7]:
jar = project/'tools/spmf.jar'
if not jar.exists():
    try:
        run_module('scripts.setup_spmf')
    except subprocess.CalledProcessError:
        print('Không tự tải được SPMF. Chạy ô tải file spmf.jar thủ công ngay dưới đây.')
print('Đã có SPMF jar?', jar.is_file())

Đã có SPMF jar? True


**SPMF trên IDE:** Ô tiếp theo kiểm tra `tools/spmf.jar`. Nếu chưa có, chạy `py -3.13 -m scripts.setup_spmf` trong Terminal rồi thử lại.


In [8]:
assert jar.is_file(), (
    'Không tìm thấy tools/spmf.jar. Chạy `py -3.13 -m scripts.setup_spmf` '
    'trong Terminal, hoặc tải SPMF jar chính thức vào tools/spmf.jar.')
print('LOCAL SPMF JAR:', jar)


LOCAL SPMF JAR: D:\gg\Group_9-Frequent-Itemset-Mining-main\tools\spmf.jar


In [9]:
import shutil
import re
import subprocess

java = shutil.which('java')
if not java:
    raise RuntimeError('Chưa tìm thấy Java trên PATH. Hãy cài Java 25 trở lên và mở lại VS Code.')
version = subprocess.run([java, '-version'], capture_output=True,
                         text=True, check=True)
java_text = version.stderr + version.stdout
print(java_text)
m = re.search(r'version\s+"?(\d+)', java_text)
if m and int(m.group(1)) < 25:
    raise RuntimeError('SPMF jar hiện tại yêu cầu Java 25 trở lên. Java đang dùng quá cũ.')
print('LOCAL Java:', java)


openjdk version "25.0.5.1" 2026-11-17 LTS
OpenJDK Runtime Environment Temurin-25.0.5.1+2 (build 25.0.5.1+2-LTS)
OpenJDK 64-Bit Server VM Temurin-25.0.5.1+2 (build 25.0.5.1+2-LTS, mixed mode, sharing)

LOCAL Java: C:\Program Files\Eclipse Adoptium\jdk-25.0.5.10002-hotspot\bin\java.EXE


In [10]:
import csv
assert jar.is_file(), 'Thiếu tools/spmf.jar'
run_module('scripts.run_spmf_fixed', '--jar', 'tools/spmf.jar')
with (project / 'results/spmf_comparison.csv').open(
        encoding='utf-8-sig', newline='') as f:
    rows = list(csv.DictReader(f))
matched = [r for r in rows if r['status'] == 'ok'
           and r['exact_match'].lower() == 'true']
print(f'SPMF EXACT MATCH: {len(matched)}/{len(rows)}')
assert len(rows) == len(matched) == 25, 'Bảng đối chiếu SPMF chưa đủ 25/25 khớp'


>>> c:\Users\aer1al\AppData\Local\Programs\Python\Python312\python.exe -m scripts.run_spmf_fixed --jar tools/spmf.jar
SPMF EXACT MATCH: 25/25


## Bước 6 — Hai ứng dụng (Chương 3)

- Ứng dụng 1: ECommerce từ SPMF → giỏ hàng → luật support, confidence, lift → top 10.
- Ứng dụng 2: UCI Optdigits → patches → k-means NumPy → visual words → Eclat.

Đầu ra tạo bằng **mã Eclat của project**. Dữ liệu ứng dụng không được thay bằng toy khi báo cáo kết quả thật.

In [11]:
run_module('scripts.market_app')
run_module('scripts.visual_app')
for filename in ('market_top10.csv','visual_patterns.csv'):
    with (project/'results'/filename).open(encoding='utf-8-sig') as handle:
        print('\n',filename)
        print(''.join(handle.readlines()[:7]))

>>> c:\Users\aer1al\AppData\Local\Programs\Python\Python312\python.exe -m scripts.market_app
>>> c:\Users\aer1al\AppData\Local\Programs\Python\Python312\python.exe -m scripts.visual_app

 market_top10.csv
X,Y,support,confidence,lift
DOLLY GIRL LUNCH BOX,SPACEBOY LUNCH BOX,0.01209,0.70155,33.88941
SPACEBOY LUNCH BOX,DOLLY GIRL LUNCH BOX,0.01209,0.58387,33.88941
PINK REGENCY TEACUP AND SAUCER,GREEN REGENCY TEACUP AND SAUCER + ROSES REGENCY TEACUP AND SAUCER,0.01449,0.65957,32.81438
GREEN REGENCY TEACUP AND SAUCER + ROSES REGENCY TEACUP AND SAUCER,PINK REGENCY TEACUP AND SAUCER,0.01449,0.72093,32.81438
GREEN REGENCY TEACUP AND SAUCER,PINK REGENCY TEACUP AND SAUCER + ROSES REGENCY TEACUP AND SAUCER,0.01449,0.52927,32.61643
PINK REGENCY TEACUP AND SAUCER + ROSES REGENCY TEACUP AND SAUCER,GREEN REGENCY TEACUP AND SAUCER,0.01449,0.893,32.61643


 visual_patterns.csv
visual_words,support_count,support_fraction
9,600,1.000000
6 9,448,0.746667
6,448,0.746667
0 9,404,0.673333
0,404,0.673333
9 11,

## Bước 7 — Tạo biểu đồ, bảng báo cáo và kiểm tra trạng thái

`docs/generated/experiment_data.tex` được sinh **từ CSV thực tế**; file này chưa tự thay tất cả phần nhận xét trong `docs/Report.tex`. Hãy giải thích kết quả của bạn, rồi biên dịch PDF tại VS Code khi có XeLaTeX.

In [12]:
run_module('scripts.plot_results')
run_module('scripts.generate_report_tables')
run_module('scripts.check_submission')
print('Đường dẫn kết quả:', project / 'results')


>>> c:\Users\aer1al\AppData\Local\Programs\Python\Python312\python.exe -m scripts.plot_results
>>> c:\Users\aer1al\AppData\Local\Programs\Python\Python312\python.exe -m scripts.generate_report_tables
>>> c:\Users\aer1al\AppData\Local\Programs\Python\Python312\python.exe -m scripts.check_submission
Đường dẫn kết quả: D:\gg\Group_9-Frequent-Itemset-Mining-main\results


## Bước 7B — Kiểm chứng Eclat bằng Julia

Bên cạnh phiên bản Python, nhóm cài đặt độc lập thuật toán Eclat bằng Julia với hai cách biểu diễn tập TID: `Set` và `BitVector`.

Phần này kiểm thử chương trình Julia, chạy benchmark trên bốn bộ dữ liệu và đối chiếu chính xác từng frequent itemset cùng giá trị support với SPMF.

**Lưu ý:** Thời gian thực thi của Julia, Python và SPMF được đo theo những giao thức khác nhau. Do đó, kết quả chỉ mang tính tham khảo, không dùng để kết luận tốc độ thuần thuật toán khi chưa thống nhất điều kiện thực nghiệm.

In [13]:
ROOT: Path = project.resolve()

if not (ROOT / 'julia/EclatJulia.jl').is_file():
    raise FileNotFoundError('Thiếu mã nguồn Julia: julia/EclatJulia.jl')

os.chdir(ROOT)
print('Project:', ROOT)
print('Julia source:', ROOT / 'julia/EclatJulia.jl')


Project: D:\gg\Group_9-Frequent-Itemset-Mining-main
Julia source: D:\gg\Group_9-Frequent-Itemset-Mining-main\julia\EclatJulia.jl


In [14]:
import shutil
import subprocess

def find_julia() -> str:
    executable = shutil.which('julia')
    if executable is None:
        raise RuntimeError('Julia chưa có trên PATH. Cài Julia rồi mở lại VS Code.')
    return executable

JULIA: str = find_julia()
subprocess.run([JULIA, '--version'], check=True)


RuntimeError: Julia chưa có trên PATH. Cài Julia rồi mở lại VS Code.

In [ ]:

import subprocess

print("=== JULIA CORRECTNESS TESTS ===")

subprocess.run(
    [JULIA, "--startup-file=no", "julia/test_eclat.jl"],
    cwd=ROOT,
    check=True
)


In [ ]:

import subprocess
import sys

print("=== JULIA BENCHMARK ===")

subprocess.run(
    [JULIA, "--startup-file=no", "julia/benchmark_julia.jl"],
    cwd=ROOT,
    check=True
)

print("=== EXPORT JULIA ITEMSETS ===")

subprocess.run(
    [JULIA, "--startup-file=no", "julia/export_itemsets.jl"],
    cwd=ROOT,
    check=True
)

print("=== JULIA VS SPMF EXACT MATCH ===")

subprocess.run(
    [sys.executable, "-m", "scripts.verify_julia_spmf"],
    cwd=ROOT,
    check=True
)

import json, platform
from datetime import datetime, timezone
info = {
    'run_utc': datetime.now(timezone.utc).isoformat(),
    'platform': platform.platform(),
    'julia': subprocess.check_output([JULIA, '--version'], text=True).strip(),
    'colab': Path('/content').is_dir()
}
(ROOT / 'results/julia_execution_environment.json').write_text(
    json.dumps(info, indent=2), encoding='utf-8')
print('Julia environment:', info)


In [ ]:
preview('julia_benchmark.csv')
print((ROOT / 'results/julia_execution_environment.json').read_text(encoding='utf-8'))


## Bước 8 — Xem kết quả trên LOCAL

Kết quả lưu trực tiếp trong thư mục `results/` của project. Không tạo ZIP để upload Colab.


In [ ]:
import csv

expected = ('benchmark.csv', 'rss_memory.csv', 'scalability.csv',
            'length_experiment.csv', 'spmf_comparison.csv',
            'julia_benchmark.csv', 'market_top10.csv', 'visual_patterns.csv')
for name in expected:
    path = project / 'results' / name
    assert path.is_file(), 'Thiếu kết quả: ' + str(path)
    with path.open(encoding='utf-8-sig', newline='') as f:
        rows = list(csv.DictReader(f))
    print(f'{name}: {len(rows)} records')

figs = sorted((project / 'results').rglob('*.png'))
print(f'\nBiểu đồ PNG trong results/: {len(figs)}')
for p in figs:
    print('  ', p.relative_to(project))
print('\nLOCAL results:', project / 'results')
